In [0]:
spark

A. Data ingestion and exploration

In [0]:
CATALOG = "workspace"
SCHEMA = "theme_park_operations"

BASE_PATH = "/Volumes/workspace/theme_park_operations/themepark_files"

prefix = f"{CATALOG}.{SCHEMA}"

print("Project:", prefix)
print("File location:", BASE_PATH)

Project: workspace.theme_park_operations
File location: /Volumes/workspace/theme_park_operations/themepark_files


In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

entry_schema = StructType([
    StructField("entry_id", IntegerType(), True),
    StructField("visitor_id", StringType(), True),
    StructField("attraction_id", StringType(), True),
    StructField("visit_date", StringType(), True),
    StructField("entry_time", StringType(), True),
    StructField("wait_minutes", IntegerType(), True),
    StructField("ride_minutes", IntegerType(), True),
    StructField("ticket_type", StringType(), True),
    StructField("ticket_price", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("booking_channel", StringType(), True)
])

maintenance_schema = StructType([
    StructField("maintenance_id", StringType(), True),
    StructField("attraction_id", StringType(), True),
    StructField("maintenance_date", StringType(), True),
    StructField("issue_type", StringType(), True),
    StructField("downtime_minutes", IntegerType(), True),
    StructField("maintenance_status", StringType(), True)
])

In [0]:
entries = (
    spark.read
    .option("header", "true")
    .schema(entry_schema)
    .csv(f"{BASE_PATH}/park_entries.csv")
)

attractions = spark.read.json(
    f"{BASE_PATH}/attractions.json"
)

maintenance = (
    spark.read
    .option("header", "true")
    .schema(maintenance_schema)
    .csv(f"{BASE_PATH}/attraction_maintenance.csv")
)

changes = (
    spark.read
    .option("header", "true")
    .schema(entry_schema)
    .csv(f"{BASE_PATH}/park_entry_changes.csv")
)

In [0]:
entries.write.format("delta").mode("overwrite").saveAsTable(
    f"{prefix}.tp_entries_raw"
)

attractions.write.format("delta").mode("overwrite").saveAsTable(
    f"{prefix}.tp_attractions"
)

maintenance.write.format("delta").mode("overwrite").saveAsTable(
    f"{prefix}.tp_maintenance"
)

changes.write.format("delta").mode("overwrite").saveAsTable(
    f"{prefix}.tp_entry_changes"
)

print("All four Delta tables created successfully!")

All four Delta tables created successfully!


In [0]:
tables = [
    "tp_entries_raw",
    "tp_attractions",
    "tp_maintenance",
    "tp_entry_changes"
]

for table in tables:
    df = spark.table(f"{prefix}.{table}")
    print(table, ":", df.count(), "rows")

tp_entries_raw : 121 rows
tp_attractions : 10 rows
tp_maintenance : 28 rows
tp_entry_changes : 10 rows


B. Cleaning and transforming with PySpark

In [0]:
entries = spark.table("workspace.theme_park_operations.tp_entries_raw")

attractions = spark.table("workspace.theme_park_operations.tp_attractions")

In [0]:
from pyspark.sql.functions import col

missing_records = entries.filter(
    col("ticket_price").isNull() |
    col("booking_channel").isNull()
)

display(missing_records)

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1016,V563,A10,2026-10-08,15:10:00,30,10,Standard,null,Completed,Kiosk
1040,V514,A03,2026-10-01,19:10:00,5,15,Standard,400.0,Completed,null
1071,V553,A07,2026-10-03,11:10:00,5,20,Standard,null,Completed,Partner
1087,V537,A01,2026-10-11,10:00:00,75,4,Standard,350.0,Completed,null


In [0]:
from pyspark.sql.functions import when

cleaned_entries = entries.withColumn(
    "booking_channel",
    when(
        col("booking_channel").isNull(),
        "Unknown"
    ).otherwise(col("booking_channel"))
)

display(cleaned_entries)

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,Unknown
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk


In [0]:
invalid_records = entries.filter(
    (col("wait_minutes") < 0) |
    (col("ride_minutes") == 0) |
    (col("ticket_price") <= 0)
)

display(invalid_records)

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1021,V527,A03,2026-10-02,13:00:00,-8,8,Express,650.0,Completed,Web
1036,V527,A06,2026-10-04,14:40:00,40,0,Standard,400.0,Completed,Kiosk
1058,V516,A02,2026-10-08,19:10:00,30,4,Standard,0.0,Completed,App
1073,V501,A03,2026-10-07,18:50:00,-3,6,Express,650.0,Completed,App


In [0]:
duplicates = (
    entries.groupBy("entry_id")
    .count()
    .filter(col("count") > 1)
)

display(duplicates)

entry_id,count
1032,2


In [0]:
deduplicated_entries = entries.dropDuplicates(["entry_id"])

display(deduplicated_entries)

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk


In [0]:
print("Original rows:", entries.count())

print("After removing duplicates:", deduplicated_entries.count())

Original rows: 121
After removing duplicates: 120


In [0]:
unknown_attractions = entries.join(
    attractions,
    on="attraction_id",
    how="left_anti"
)

display(unknown_attractions)

attraction_id,entry_id,visitor_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
A99,1092,V566,2026-10-03,15:40:00,75,20,Standard,400.0,Completed,Kiosk


In [0]:
from pyspark.sql.functions import when

valid_waits = entries.filter(
    col("wait_minutes") >= 0
)

wait_category_df = valid_waits.withColumn(
    "wait_category",
    when(col("wait_minutes") <= 20, "Short")
    .when(col("wait_minutes") <= 45, "Medium")
    .otherwise("Long")
)

display(wait_category_df)

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel,wait_category
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App,Short
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App,Long
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web,Medium
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web,Short
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk,Short
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web,Short
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web,Short
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk,Medium
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null,Short
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk,Short


In [0]:
display(
    wait_category_df.groupBy("wait_category").count()
)

wait_category,count
Short,57
Long,31
Medium,31


In [0]:
joined_entries = entries.join(
    attractions,
    on="attraction_id",
    how="inner"
)

display(joined_entries)

attraction_id,entry_id,visitor_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel,attraction_name,capacity_per_hour,category,features,is_active,location,min_height_cm,opened_year
A08,1001,V515,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App,Forest Flyers,120,Adventure,"List(Outdoor, Interactive)",true,"List(W1, West)",115,2023
A01,1002,V535,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App,Sky Spinner,140,Thrill,"List(Outdoor, High Speed)",true,"List(N1, North)",120,2018
A04,1003,V547,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web,Moonlight Train,180,Family,"List(Scenic, Family Friendly)",true,"List(W3, West)",0,2012
A05,1004,V517,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web,Meteor Drop,100,Thrill,"List(High Speed, Outdoor)",true,"List(N2, North)",135,2022
A07,1005,V537,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk,Crystal Maze,160,Adventure,"List(Indoor, Puzzle)",true,"List(S3, South)",90,2021
A08,1006,V549,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web,Forest Flyers,120,Adventure,"List(Outdoor, Interactive)",true,"List(W1, West)",115,2023
A08,1007,V529,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web,Forest Flyers,120,Adventure,"List(Outdoor, Interactive)",true,"List(W1, West)",115,2023
A06,1008,V521,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk,Pirate Voyage,130,Water,"List(Water Ride, Themed)",true,"List(E1, East)",100,2019
A02,1009,V510,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null,Coral Carousel,200,Family,"List(Indoor, Family Friendly)",true,"List(E2, East)",0,2015
A02,1010,V560,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk,Coral Carousel,200,Family,"List(Indoor, Family Friendly)",true,"List(E2, East)",0,2015


In [0]:
from pyspark.sql.functions import col

attraction_details = joined_entries.select(
    "entry_id",
    "visitor_id",
    "attraction_id",
    "attraction_name",
    "category",
    col("location.zone").alias("zone"),
    col("features").alias("features"),
    "ticket_type",
    "ticket_price",
    "wait_minutes"
)

display(attraction_details)

entry_id,visitor_id,attraction_id,attraction_name,category,zone,features,ticket_type,ticket_price,wait_minutes
1001,V515,A08,Forest Flyers,Adventure,West,"List(Outdoor, Interactive)",Standard,350.0,15
1002,V535,A01,Sky Spinner,Thrill,North,"List(Outdoor, High Speed)",Standard,350.0,65
1003,V547,A04,Moonlight Train,Family,West,"List(Scenic, Family Friendly)",Standard,350.0,40
1004,V517,A05,Meteor Drop,Thrill,North,"List(High Speed, Outdoor)",Express,650.0,20
1005,V537,A07,Crystal Maze,Adventure,South,"List(Indoor, Puzzle)",Standard,350.0,5
1006,V549,A08,Forest Flyers,Adventure,West,"List(Outdoor, Interactive)",VIP,900.0,10
1007,V529,A08,Forest Flyers,Adventure,West,"List(Outdoor, Interactive)",Standard,350.0,5
1008,V521,A06,Pirate Voyage,Water,East,"List(Water Ride, Themed)",Express,650.0,40
1009,V510,A02,Coral Carousel,Family,East,"List(Indoor, Family Friendly)",Standard,350.0,10
1010,V560,A02,Coral Carousel,Family,East,"List(Indoor, Family Friendly)",Express,600.0,15


C. SQL joins and aggregations

In [0]:
entries_table = "workspace.theme_park_operations.tp_entries_raw"

attractions_table = "workspace.theme_park_operations.tp_attractions"

maintenance_table = "workspace.theme_park_operations.tp_maintenance"

In [0]:
display(spark.table(entries_table))
display(spark.table(attractions_table))
display(spark.table(maintenance_table))

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk


attraction_id,attraction_name,capacity_per_hour,category,features,is_active,location,min_height_cm,opened_year
A01,Sky Spinner,140,Thrill,"List(Outdoor, High Speed)",true,"List(N1, North)",120,2018
A02,Coral Carousel,200,Family,"List(Indoor, Family Friendly)",true,"List(E2, East)",0,2015
A03,Jungle Rapids,110,Water,"List(Water Ride, Outdoor)",true,"List(S1, South)",110,2020
A04,Moonlight Train,180,Family,"List(Scenic, Family Friendly)",true,"List(W3, West)",0,2012
A05,Meteor Drop,100,Thrill,"List(High Speed, Outdoor)",true,"List(N2, North)",135,2022
A06,Pirate Voyage,130,Water,"List(Water Ride, Themed)",true,"List(E1, East)",100,2019
A07,Crystal Maze,160,Adventure,"List(Indoor, Puzzle)",true,"List(S3, South)",90,2021
A08,Forest Flyers,120,Adventure,"List(Outdoor, Interactive)",true,"List(W1, West)",115,2023
A09,Galaxy Theater,260,Show,"List(Indoor, Live Show)",true,"List(N3, North)",0,2017
A10,Mini Explorers,190,Family,"List(Kids, Indoor)",true,"List(E3, East)",0,2016


maintenance_id,attraction_id,maintenance_date,issue_type,downtime_minutes,maintenance_status
M001,A05,2026-10-15,Cleaning,90,Scheduled
M002,A08,2026-10-05,Mechanical,60,Completed
M003,A02,2026-10-12,Mechanical,0,Scheduled
M004,A09,2026-10-18,Electrical,0,Open
M005,A05,2026-10-12,Electrical,0,Scheduled
M006,A04,2026-10-07,Water System,45,Completed
M007,A10,2026-10-20,Mechanical,120,Completed
M008,A04,2026-10-04,Water System,90,Open
M009,A08,2026-10-19,Safety Check,30,Scheduled
M010,A02,2026-10-15,Safety Check,0,Open


In [0]:
query = """
SELECT *
FROM workspace.theme_park_operations.tp_entries_raw
WHERE status = 'Completed'
AND ticket_type IN ('Standard', 'Express')
AND ticket_price BETWEEN 0.01 AND 100000
"""

display(spark.sql(query))

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1013,V506,A09,2026-10-11,10:20:00,15,20,Standard,350.0,Completed,Web
1015,V529,A05,2026-10-01,13:40:00,10,15,Standard,400.0,Completed,Kiosk
1017,V501,A10,2026-10-06,17:50:00,40,8,Standard,350.0,Completed,Partner


In [0]:
query = """
SELECT
    attraction_id,
    COUNT(*) AS completed_entries,
    SUM(ticket_price) AS total_revenue
FROM workspace.theme_park_operations.tp_entries_raw
WHERE status = 'Completed'
AND ticket_price > 0
GROUP BY attraction_id
ORDER BY total_revenue DESC
"""

display(spark.sql(query))

attraction_id,completed_entries,total_revenue
A10,13,6050.0
A05,10,5850.0
A04,12,5450.0
A07,9,4750.0
A06,9,4600.0
A03,8,4500.0
A01,9,4200.0
A08,8,3600.0
A09,7,3200.0
A02,5,2000.0


In [0]:
query = """
SELECT
    a.category,
    AVG(e.wait_minutes) AS average_waiting_time
FROM workspace.theme_park_operations.tp_entries_raw e
INNER JOIN workspace.theme_park_operations.tp_attractions a
ON e.attraction_id = a.attraction_id
WHERE e.wait_minutes >= 0
GROUP BY a.category
ORDER BY average_waiting_time DESC
"""

display(spark.sql(query))

category,average_waiting_time
Show,35.90909090909091
Adventure,33.958333333333336
Family,31.463414634146343
Thrill,30.5
Water,25.227272727272727


In [0]:
query = """
SELECT
    a.attraction_id,
    a.attraction_name,
    a.category,
    SUM(m.downtime_minutes) AS total_downtime
FROM workspace.theme_park_operations.tp_maintenance m
INNER JOIN workspace.theme_park_operations.tp_attractions a
ON m.attraction_id = a.attraction_id
WHERE m.maintenance_status = 'Completed'
GROUP BY
    a.attraction_id,
    a.attraction_name,
    a.category
ORDER BY total_downtime DESC
"""

display(spark.sql(query))

attraction_id,attraction_name,category,total_downtime
A02,Coral Carousel,Family,240
A06,Pirate Voyage,Water,135
A01,Sky Spinner,Thrill,120
A10,Mini Explorers,Family,120
A07,Crystal Maze,Adventure,120
A04,Moonlight Train,Family,75
A08,Forest Flyers,Adventure,60
A03,Jungle Rapids,Water,30
A09,Galaxy Theater,Show,0


In [0]:
query = """
SELECT
    a.location.zone AS park_zone,
    SUM(e.ticket_price) AS total_revenue,
    COUNT(*) AS completed_entries
FROM workspace.theme_park_operations.tp_entries_raw e
INNER JOIN workspace.theme_park_operations.tp_attractions a
ON e.attraction_id = a.attraction_id
WHERE e.status = 'Completed'
AND e.ticket_price > 0
GROUP BY a.location.zone
ORDER BY total_revenue DESC
"""

display(spark.sql(query))

park_zone,total_revenue,completed_entries
North,13250.0,26
East,12650.0,27
South,9250.0,17
West,9050.0,20


D. Delta table creation and CRUD

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.theme_park_operations.tp_entries_work
USING DELTA
AS
SELECT *
FROM workspace.theme_park_operations.tp_entries_raw
WHERE entry_id IS NOT NULL
AND attraction_id IS NOT NULL
AND wait_minutes >= 0
AND ride_minutes > 0
AND ticket_price > 0;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * 
FROM workspace.theme_park_operations.tp_entries_work;

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk


In [0]:
entries = spark.table(
    "workspace.theme_park_operations.tp_entries_work"
)

entries.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.theme_park_operations.tp_entries_backup"
    )

print("Backup Delta table created successfully")

Backup Delta table created successfully


In [0]:
display(
    spark.table(
        "workspace.theme_park_operations.tp_entries_backup"
    )
)

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk


In [0]:
%sql
INSERT INTO workspace.theme_park_operations.tp_entries_work
VALUES
(1001, 'V1001', 'A01', '2026-11-11', '10:30:00', 15, 5, 'Standard', 450.0, 'Completed', 'Online'),
(1002, 'V1002', 'A02', '2026-11-11', '11:00:00', 25, 8, 'Express', 650.0, 'Completed', 'Mobile');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
SELECT *
FROM workspace.theme_park_operations.tp_entries_work
WHERE entry_id IN (1001, 1002);

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1001,V1001,A01,2026-11-11,10:30:00,15,5,Standard,450.0,Completed,Online
1002,V1002,A02,2026-11-11,11:00:00,25,8,Express,650.0,Completed,Mobile


In [0]:
%sql
UPDATE workspace.theme_park_operations.tp_entries_work
SET
    status = 'Completed',
    ticket_price = 500.0
WHERE entry_id = 1001;

num_affected_rows
2


In [0]:
%sql
SELECT *
FROM workspace.theme_park_operations.tp_entries_work
WHERE entry_id = 1001;

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,500.0,Completed,App
1001,V1001,A01,2026-11-11,10:30:00,15,5,Standard,500.0,Completed,Online


In [0]:
%sql
SELECT *
FROM workspace.theme_park_operations.tp_entries_work
WHERE status = 'Cancelled';

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk
1011,V536,A02,2026-10-06,16:40:00,25,8,Standard,350.0,Cancelled,App
1012,V543,A07,2026-10-13,12:20:00,25,20,Express,600.0,Cancelled,Web
1020,V534,A06,2026-10-07,17:10:00,10,12,Standard,400.0,Cancelled,Kiosk
1031,V520,A10,2026-10-14,10:20:00,50,4,Standard,350.0,Cancelled,Partner
1035,V511,A03,2026-10-10,19:50:00,65,15,Express,650.0,Cancelled,Partner
1038,V554,A07,2026-10-03,19:00:00,20,20,Standard,350.0,Cancelled,Partner
1043,V542,A10,2026-10-03,12:10:00,40,8,Express,600.0,Cancelled,Kiosk
1051,V543,A03,2026-10-14,13:40:00,40,10,Standard,400.0,Cancelled,Partner
1052,V501,A09,2026-10-09,15:00:00,30,6,Standard,350.0,Cancelled,Partner


In [0]:
%sql DESCRIBE EXTENDED workspace.theme_park_operations.tp_entries_work;


col_name,data_type,comment
entry_id,int,null
visitor_id,string,null
attraction_id,string,null
visit_date,string,null
entry_time,string,null
wait_minutes,int,null
ride_minutes,int,null
ticket_type,string,null
ticket_price,double,null
status,string,null


E. MERGE / UPSERT and SCD

In [0]:
changes = spark.table("workspace.theme_park_operations.tp_entry_changes")

display(changes)
print("Source records:", changes.count())

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,400.0,Completed,App
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,675.0,Completed,App
1040,V514,A03,2026-10-01,19:10:00,5,15,Standard,500.0,Completed,App
1068,V531,A10,2026-10-15,16:20:00,30,4,Express,725.0,Completed,App
1095,V563,A07,2026-10-09,11:20:00,75,4,Standard,500.0,Completed,App
1121,V700,A01,2026-10-20,11:00:00,15,8,Standard,350.0,Completed,Web
1122,V701,A02,2026-10-20,12:00:00,20,10,Express,600.0,Completed,Web
1123,V702,A03,2026-10-20,13:00:00,25,12,Standard,350.0,Completed,Web
1124,V703,A04,2026-10-20,14:00:00,30,14,Express,600.0,Completed,Web
1125,V704,A05,2026-10-20,15:00:00,35,16,Standard,350.0,Completed,Web


Source records: 10


In [0]:
work = spark.table("workspace.theme_park_operations.tp_entries_work")

display(work)
print("Working table records:", work.count())

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,500.0,Completed,App
1001,V1001,A01,2026-11-11,10:30:00,15,5,Standard,500.0,Completed,Online
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null


Working table records: 117


In [0]:
%sql MERGE INTO workspace.theme_park_operations.tp_entries_work AS target
USING workspace.theme_park_operations.tp_entry_changes AS source
ON target.entry_id = source.entry_id

WHEN MATCHED THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
10,5,0,5


In [0]:
%sql
SELECT *
FROM workspace.theme_park_operations.tp_entries_work
ORDER BY entry_id;

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V1001,A01,2026-11-11,10:30:00,15,5,Standard,500.0,Completed,Online
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,500.0,Completed,App
1002,V1002,A02,2026-11-11,11:00:00,25,8,Express,650.0,Completed,Mobile
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,400.0,Completed,App
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk


In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM workspace.theme_park_operations.tp_entries_work;

total_records
122


In [0]:
%sql
SELECT entry_id, COUNT(*) AS record_count
FROM workspace.theme_park_operations.tp_entries_work
GROUP BY entry_id
HAVING COUNT(*) > 1;

entry_id,record_count
1002,2
1001,2
1032,2


F. Delta internals, Time Travel, and optimization

In [0]:
%sql DESCRIBE HISTORY workspace.theme_park_operations.tp_entries_work

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-11-11T06:00:29.000Z,75635105658499,bhavani.k0408@gmail.com,MERGE,"Map(predicate -> [""(entry_id#18386 = entry_id#18397)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(1993346764184000),afeb2cb0-003f-44ee-b74f-bd90b78469e1,1111-043128-rsnm06ug-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 4330, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 3371, materializeSourceTimeMs -> 6, numTargetRowsInserted -> 5, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1057, numTargetRowsUpdated -> 5, numOutputRows -> 10, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 10, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2234)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13
3,2026-11-11T05:48:35.000Z,75635105658499,bhavani.k0408@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1993346764184000),1d39322d-eec2-4617-9e24-5cecc7aace33,1111-043128-rsnm06ug-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 11727, p25FileSize -> 5527, numDeletionVectorsRemoved -> 2, minFileSize -> 5527, numAddedFiles -> 1, maxFileSize -> 5527, p75FileSize -> 5527, p50FileSize -> 5527, numAddedBytes -> 5527)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13
2,2026-11-11T05:48:32.000Z,75635105658499,bhavani.k0408@gmail.com,UPDATE,"Map(predicate -> [""(entry_id#16505 = 1001)""])",null,List(1993346764184000),1d39322d-eec2-4617-9e24-5cecc7aace33,1111-043128-rsnm06ug-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 2, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3742, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1865, numAddedFiles -> 1, numUpdatedRows -> 2, numAddedBytes -> 3974, rewriteTimeMs -> 1844)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13
1,2026-11-11T05:46:13.000Z,75635105658499,bhavani.k0408@gmail.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> true, mode -> Append, clusteringOnWriteStatus -> null)",null,List(1993346764184000),0dc029f3-3b8d-4a9b-bcee-817bcffa8f1b,1111-043128-rsnm06ug-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 3190)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13
0,2026-11-11T05:41:11.000Z,75635105658499,bhavani.k0408@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""3ba369f1-3fff-4678-85cd-b218cd346c64"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-22527978-ea87-4da6-a21d-860f8dc79a99"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-d258d00e-c089-4450-a812-193f11bce6b1""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1993346764184000),1160ea81-19d3-49ff-8c74-ea144

In [0]:
%sql
SELECT *
FROM workspace.theme_park_operations.tp_entries_work
VERSION AS OF 0;

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk


In [0]:
%sql
DESCRIBE DETAIL workspace.theme_park_operations.tp_entries_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,2bd58648-205b-4b58-a891-d5ada14035a9,workspace.theme_park_operations.tp_entries_work,null,,2026-11-11T05:41:08.684Z,2026-11-11T06:00:29.000Z,List(),List(),2,9857,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> 3ba369f1-3fff-4678-85cd-b218cd346c64, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-22527978-ea87-4da6-a21d-860f8dc79a99, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-d258d00e-c089-4450-a812-193f11bce6b1)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 5, numDeletionVectors -> 1)",true


In [0]:
%sql
RESTORE TABLE workspace.theme_park_operations.tp_entries_work
TO VERSION AS OF 0;

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
4563,1,2,1,9857,4563


G. Views and Parquet

In [0]:
source_df = spark.table(
    "workspace.theme_park_operations.tp_entries_raw"
)

display(source_df)
print("Initial records:", source_df.count())

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk


Initial records: 121


In [0]:
stream_df = spark.readStream.table(
    "workspace.theme_park_operations.tp_entries_raw"
)

print("Is streaming:", stream_df.isStreaming)

Is streaming: True


In [0]:
checkpoint_path = "/Volumes/workspace/theme_park_operations/themepark_files/checkpoints/tp_entries_stream"

query = (
    stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("workspace.theme_park_operations.tp_entries_stream")
)

In [0]:
stream_output = spark.table(
    "workspace.theme_park_operations.tp_entries_stream"
)

display(stream_output)
print("Streaming records:", stream_output.count())

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk


Streaming records: 121


In [0]:
%sql
SELECT COUNT(*) AS total_stream_records
FROM workspace.theme_park_operations.tp_entries_stream;

total_stream_records
121


In [0]:
incremental_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/theme_park_operations/themepark_files/park_entries_incremental.csv")
)

display(incremental_df)
print("Incremental records:", incremental_df.count())

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1130,V800,A01,2026-11-11,10:00:00,10,5,Standard,350,Pending,App
1131,V801,A02,2026-11-11,11:10:00,15,8,Express,600,Completed,Web
1132,V802,A03,2026-11-11,12:20:00,20,11,VIP,900,Completed,App
1133,V803,A04,2026-11-11,13:30:00,25,14,Standard,350,Completed,Web
1134,V804,A05,2026-11-11,14:40:00,30,17,Express,600,Pending,App
1135,V805,A06,2026-11-11,15:50:00,35,5,VIP,900,Completed,Web
1136,V806,A07,2026-11-12,16:00:00,40,8,Standard,350,Completed,App
1137,V807,A08,2026-11-12,17:10:00,10,11,Express,600,Completed,Web
1138,V808,A09,2026-11-12,18:20:00,15,14,VIP,900,Pending,App
1139,V809,A10,2026-11-12,10:30:00,20,17,Standard,350,Completed,Web


Incremental records: 12


In [0]:
incremental_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.theme_park_operations.tp_entries_raw")

In [0]:
print(
    "Updated raw count:",
    spark.table("workspace.theme_park_operations.tp_entries_raw").count()
)

Updated raw count: 133


H. Structured Streaming

In [0]:
entries = spark.table(
    "workspace.theme_park_operations.tp_entries_raw"
)

print("Total records:", entries.count())
display(entries)

Total records: 133


entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk


In [0]:
from pyspark.sql.functions import col, sum, when

entries.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in entries.columns
]).show()

+--------+----------+-------------+----------+----------+------------+------------+-----------+------------+------+---------------+
|entry_id|visitor_id|attraction_id|visit_date|entry_time|wait_minutes|ride_minutes|ticket_type|ticket_price|status|booking_channel|
+--------+----------+-------------+----------+----------+------------+------------+-----------+------------+------+---------------+
|       0|         0|            0|         0|         0|           0|           0|          0|           2|     0|              3|
+--------+----------+-------------+----------+----------+------------+------------+-----------+------------+------+---------------+



In [0]:
from pyspark.sql.functions import count

duplicates = (
    entries.groupBy("entry_id")
    .agg(count("*").alias("record_count"))
    .filter(col("record_count") > 1)
)

display(duplicates)

entry_id,record_count
1032,2


In [0]:
%sql OPTIMIZE workspace.theme_park_operations.tp_entries_work

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1794379426239, 1794379428496, 8, 0, null, List(0, 0), null, 11, 11, 0, 0, null, null, 0)"


In [0]:
%sql
DESCRIBE HISTORY workspace.theme_park_operations.tp_entries_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-11-11T06:24:12.000Z,75635105658499,bhavani.k0408@gmail.com,RESTORE,"Map(version -> 0, timestamp -> null)",null,List(1993346764184000),da211cdf-7e2c-4bb0-9f0b-006ac6a73c52,1111-043128-rsnm06ug-v2n,4,Serializable,false,"Map(numRestoredFiles -> 1, removedFilesSize -> 9857, numRemovedFiles -> 2, restoredFilesSize -> 4563, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 1, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 4563)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13
4,2026-11-11T06:00:29.000Z,75635105658499,bhavani.k0408@gmail.com,MERGE,"Map(predicate -> [""(entry_id#18386 = entry_id#18397)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(1993346764184000),afeb2cb0-003f-44ee-b74f-bd90b78469e1,1111-043128-rsnm06ug-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 4330, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 3371, materializeSourceTimeMs -> 6, numTargetRowsInserted -> 5, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1057, numTargetRowsUpdated -> 5, numOutputRows -> 10, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 10, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2234)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13
3,2026-11-11T05:48:35.000Z,75635105658499,bhavani.k0408@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1993346764184000),1d39322d-eec2-4617-9e24-5cecc7aace33,1111-043128-rsnm06ug-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 11727, p25FileSize -> 5527, numDeletionVectorsRemoved -> 2, minFileSize -> 5527, numAddedFiles -> 1, maxFileSize -> 5527, p75FileSize -> 5527, p50FileSize -> 5527, numAddedBytes -> 5527)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13
2,2026-11-11T05:48:32.000Z,75635105658499,bhavani.k0408@gmail.com,UPDATE,"Map(predicate -> [""(entry_id#16505 = 1001)""])",null,List(1993346764184000),1d39322d-eec2-4617-9e24-5cecc7aace33,1111-043128-rsnm06ug-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 2, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3742, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1865, numAddedFiles -> 1, numUpdatedRows -> 2, numAddedBytes -> 3974, rewriteTimeMs -> 1844)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13
1,2026-11-11T05:46:13.000Z,75635105658499,bhavani.k0408@gmail.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> true, mode -> Append, clusteringOnWriteStatus -> null)",null,List(1993346764184000),0dc029f3-3b8d-4a9b-bcee-817bcffa8f1b,1111-043128-rsnm06ug-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 3190)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13
0,2026-11-11T05:41:11.000Z,75635105658499,bhavani.k0408@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""3ba369f1-3fff-4678-85cd-b

I. Lakeflow / DLT and incremental loading

In [0]:
from pyspark.sql.functions import (
    count, sum, avg, round, col
)

entries = spark.table(
    "workspace.theme_park_operations.tp_entries_raw"
)

attractions = spark.table(
    "workspace.theme_park_operations.tp_attractions"
)

completed = entries.filter(
    col("status") == "Completed"
)

print("Total Entries:", entries.count())
print("Completed Entries:", completed.count())

print("Total Revenue:")
completed.agg(
    round(sum("ticket_price"), 2).alias("total_revenue")
).show()

print("Average Waiting Time:")
completed.agg(
    round(avg("wait_minutes"), 2).alias("average_wait")
).show()

Total Entries: 133
Completed Entries: 103
Total Revenue:
+-------------+
|total_revenue|
+-------------+
|      50150.0|
+-------------+

Average Waiting Time:
+------------+
|average_wait|
+------------+
|       30.62|
+------------+



In [0]:
final_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/theme_park_operations/themepark_files/park_entries_incremental.csv")
)

display(final_df)

print("Final incremental records:", final_df.count())

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1130,V800,A01,2026-11-11,10:00:00,10,5,Standard,350,Pending,App
1131,V801,A02,2026-11-11,11:10:00,15,8,Express,600,Completed,Web
1132,V802,A03,2026-11-11,12:20:00,20,11,VIP,900,Completed,App
1133,V803,A04,2026-11-11,13:30:00,25,14,Standard,350,Completed,Web
1134,V804,A05,2026-11-11,14:40:00,30,17,Express,600,Pending,App
1135,V805,A06,2026-11-11,15:50:00,35,5,VIP,900,Completed,Web
1136,V806,A07,2026-11-12,16:00:00,40,8,Standard,350,Completed,App
1137,V807,A08,2026-11-12,17:10:00,10,11,Express,600,Completed,Web
1138,V808,A09,2026-11-12,18:20:00,15,14,VIP,900,Pending,App
1139,V809,A10,2026-11-12,10:30:00,20,17,Standard,350,Completed,Web


Final incremental records: 12


In [0]:
final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.theme_park_operations.tp_final_incremental"
    )

In [0]:
final_table = spark.table(
    "workspace.theme_park_operations.tp_final_incremental"
)

display(final_table)

print("Final table count:", final_table.count())

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1130,V800,A01,2026-11-11,10:00:00,10,5,Standard,350,Pending,App
1131,V801,A02,2026-11-11,11:10:00,15,8,Express,600,Completed,Web
1132,V802,A03,2026-11-11,12:20:00,20,11,VIP,900,Completed,App
1133,V803,A04,2026-11-11,13:30:00,25,14,Standard,350,Completed,Web
1134,V804,A05,2026-11-11,14:40:00,30,17,Express,600,Pending,App
1135,V805,A06,2026-11-11,15:50:00,35,5,VIP,900,Completed,Web
1136,V806,A07,2026-11-12,16:00:00,40,8,Standard,350,Completed,App
1137,V807,A08,2026-11-12,17:10:00,10,11,Express,600,Completed,Web
1138,V808,A09,2026-11-12,18:20:00,15,14,VIP,900,Pending,App
1139,V809,A10,2026-11-12,10:30:00,20,17,Standard,350,Completed,Web


Final table count: 12
